# hai07 - SFT tahap 7 (v7): jenis fakta baru, v7 (perbaikan v6) (model 100M)
Input (Add Input): **hai07-sft7** (zip: kode, data jenis fakta baru, v7 (perbaikan v6) + data tahap 5 dan 4 + replay, vocab; cache token dibuat saat latih) dan **output notebook SFT tahap 5** (berkas `reports/checkpoints_100m_sft_mem2/gpt_sft_best.pt`). Pasang HANYA output tahap 5.
Settings: Accelerator = GPU T4 x2, Persistence = Files. Jalankan `SMOKE=True` dulu, lalu `False`.
Data baru v7: pertanyaan balik pada dua kalimat kembar, dua orang, frasa tempat lengkap, dua harga, rentang tanggal; ditambah famili tahap 6 (tim, divisi, peran, jam buka/tutup, mulai/selesai). Porsi data lama dinaikkan dan LR diturunkan (1e-5) karena v6 menggeser perilaku lama. Data baru: tim dan divisi, peran (ketua, wakil, sekretaris), jam buka lawan tutup, mulai lawan selesai, jumlah anggota, ruang dan lantai; banyak pengecoh sejenis dan soal tak terjawab. Dicampur data tahap 5/4 dan replay agar kemampuan lama tidak hilang.
Output: `reports/checkpoints_100m_sft_mem4/gpt_sft_best.pt`.


In [ ]:
SMOKE = True        # True: 30 step uji kecepatan; False: SFT penuh
ENV = "production_100m"
EPOCHS = 3
BATCH = 32
LR = "1e-5"
WARMUP = "50"
TIME_LIMIT_S = 5*3600


In [ ]:
import os, glob, shutil, subprocess, sys, time, threading, zipfile
!nvidia-smi
WORK = "/kaggle/working/hai07-ocd"
os.makedirs(WORK, exist_ok=True)
for root, dirs, files in os.walk("/kaggle/input"):
    if root.count(os.sep) - 2 >= 3: dirs[:] = []; continue
    print(root, "->", (dirs + files)[:8])

# Cari paket SFT tahap 7 (v7) (berisi data sft_mem4.jsonl)
SRC = None
for f in glob.glob("/kaggle/input/**/sft_mem4.jsonl", recursive=True):
    SRC = os.path.dirname(os.path.dirname(os.path.dirname(f))); break
if SRC is None:
    zips = glob.glob("/kaggle/input/**/hai07-sft7*.zip", recursive=True)
    assert zips, "hai07-sft7 tidak ditemukan; tambahkan dataset hai07-sft7 (lihat daftar di atas)"
    zipfile.ZipFile(zips[0]).extractall("/kaggle/working/sft_src")
    cand = glob.glob("/kaggle/working/sft_src/**/sft_mem4.jsonl", recursive=True)
    SRC = os.path.dirname(os.path.dirname(os.path.dirname(cand[0])))
print("sumber kode+data:", SRC)
for d in ("apps", "tools", "data"):
    shutil.rmtree(f"{WORK}/{d}", ignore_errors=True)
    shutil.copytree(os.path.join(SRC, d), f"{WORK}/{d}")
os.chdir(WORK)

ck = [f for f in glob.glob("/kaggle/input/**/gpt_sft_best.pt", recursive=True) if "checkpoints_100m_sft_mem2" in f]
assert ck, "gpt_sft_best.pt SFT tahap 5 (folder checkpoints_100m_sft_mem2) tidak ditemukan; tambahkan output notebook SFT tahap 5 sebagai Input"
BASE = ck[0]
print("bobot dasar:", BASE)
import torch; print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
CK = f"{WORK}/reports/checkpoints_100m_sft_mem4"
cmd = ["python", "-u", "tools/train_sft.py", "--env", ENV, "--device", "cuda", "--amp",
       "--init-from", BASE, "--save-dir", CK,
       "--batch-size", str(BATCH), "--lr", LR, "--warmup-steps", WARMUP,
       "--data", "data/sft/sft_mem4.jsonl", "--val-data", "data/sft/sft_mem4_val.jsonl",
       "--token-cache", "data/sft/sft_mem4_tokens_train.pt", "--val-token-cache", "data/sft/sft_mem4_tokens_val.pt"]
if SMOKE:
    cmd += ["--max-steps", "30", "--log-every", "5", "--val-every", "1000", "--val-batches", "10", "--save-every", "100000"]
else:
    cmd += ["--epochs", str(EPOCHS), "--log-every", "25", "--val-every", "200", "--save-every", "500"]
print(" ".join(cmd))
t0 = time.time()
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
def watchdog():
    time.sleep(TIME_LIMIT_S)
    if p.poll() is None:
        print("[notebook] batas waktu tercapai -> menghentikan", flush=True); p.terminate()
threading.Thread(target=watchdog, daemon=True).start()
for line in p.stdout:
    print(line, end="", flush=True)
p.wait()
print("elapsed(min):", (time.time()-t0)/60, "returncode:", p.returncode)


In [ ]:
!ls -la {CK}; du -sh /kaggle/working